In [1]:
import numpy as np

# ==========================
# precision control
# ==========================
DT = np.longdouble


def asDT(x):
    return np.asarray(x, dtype=DT)

# ==========================================
# Tiny MQMQA G^ex toy verifier (Eq. 17 style)
# FULL 2x2 set: 9 quadruplet moles
# χ uses UPDATED Eq. 22 weighting (δ_kx+δ_ky)/2
# ==========================================

# Variable ordering (9 quadruplet mole numbers)
# 3 cation pairs: AA, AB, BB
# 3 anion pairs : XX, XY, YY
vars_ = ["AA_XX","AB_XX","BB_XX",
         "AA_XY","AB_XY","BB_XY",
         "AA_YY","AB_YY","BB_YY"]
idx = {v:i for i,v in enumerate(vars_)}
M = len(vars_)

def chi_and_derivs_updated(n, anion):
    """
    UPDATED Eq. 22 style weighting for χ_{.. /kk}:

      weight_k(ab/xy) = (δ_{k x} + δ_{k y})/2
      => for k=X:  w(../XX)=1, w(../XY)=0.5, w(../YY)=0
         for k=Y:  w(../YY)=1, w(../XY)=0.5, w(../XX)=0

    Toy choice for χ:
      χ_k   = (AA contribution with the same weight_k) / (all cation pairs contribution with weight_k)
      tχ_k  = (BB contribution with the same weight_k) / (same denominator)

    This keeps χ as a ratio of linear forms: χ = (a·n)/(b·n).
    Derivatives (ratio-of-linear-sums) still work, but now b_u can be 0.5 too.
    """
    b = np.zeros(M, dtype=DT)      # denominator weights
    a_chi = np.zeros(M, dtype=DT)  # numerator weights for χ (AA)
    a_t   = np.zeros(M, dtype=DT)  # numerator weights for tχ (BB)

    if anion == "X":
        # b: all ij with XX weight 1, XY weight 0.5
        for ij in ["AA","AB","BB"]:
            b[idx[f"{ij}_XX"]] = DT("1.0")
            b[idx[f"{ij}_XY"]] = DT("0.5")
            b[idx[f"{ij}_YY"]] = DT("0.0")

        # AA numerator with same weights
        a_chi[idx["AA_XX"]] = DT("1.0")
        a_chi[idx["AA_XY"]] = DT("0.5")

        # BB numerator with same weights
        a_t[idx["BB_XX"]] = DT("1.0")
        a_t[idx["BB_XY"]] = DT("0.5")

    elif anion == "Y":
        # b: all ij with YY weight 1, XY weight 0.5
        for ij in ["AA","AB","BB"]:
            b[idx[f"{ij}_YY"]] = 1.0
            b[idx[f"{ij}_XY"]] = DT("0.5")
            b[idx[f"{ij}_XX"]] = 0.0

        a_chi[idx["AA_YY"]] = DT("1.0")
        a_chi[idx["AA_XY"]] = DT("0.5")

        a_t[idx["BB_YY"]] = DT("1.0")
        a_t[idx["BB_XY"]] = DT("0.5")

    else:
        raise ValueError("anion must be 'X' or 'Y'")

    n = asDT(n)
    B = np.dot(b, n)  # denominator
    if B <= 0:
        raise ValueError("Denominator B <= 0; choose positive n so weighted sum is positive.")

    chi  = np.dot(a_chi, n) / B
    tchi = np.dot(a_t,   n) / B

    # First derivatives for ratio of linear forms:
    #   χ_u = (a_u - χ b_u)/B
    chi_u = (a_chi - chi*b) / B
    t_u   = (a_t   - tchi*b) / B

    # Second derivatives:
    #   χ_uv = -(b_u χ_v + b_v χ_u)/B   (since a,b constant and B linear)
    chi_uv = np.zeros((M, M), dtype=DT)
    t_uv   = np.zeros((M, M), dtype=DT)
    for u in range(M):
        for v in range(M):
            chi_uv[u, v] = -(b[u]*chi_u[v] + b[v]*chi_u[u]) / B
            t_uv[u, v]   = -(b[u]*t_u[v]   + b[v]*t_u[u])   / B

    return chi, tchi, chi_u, t_u, chi_uv, t_uv

def delta_g_and_derivs(n):
    """
    Toy Δg (binary χ-form like Eq. 23) only for anion-diagonal quadruplets:
      Δg_{ij/XX} = g_{ij,XX} * χ_X * tχ_X
      Δg_{ij/YY} = g_{ij,YY} * χ_Y * tχ_Y
    and Δg for /XY quadruplets = 0 (toy choice).

    IMPORTANT: even though Δg(/XY)=0, /XY moles DO influence χ_X and χ_Y now (updated Eq. 22),
    so they still influence Δg(/XX) and Δg(/YY) via χ.
    """
    dg    = np.zeros(M, dtype=DT)
    dg_p  = np.zeros((M, M), dtype=DT)
    dg_pq = np.zeros((M, M, M), dtype=DT)

    # arbitrary toy coefficients
    g = {
        "AA_XX": DT("2000.0"), "AB_XX": DT("3000.0"), "BB_XX": DT("1500.0"),
        "AA_YY": DT("2500.0"), "AB_YY": DT("3500.0"), "BB_YY": DT("1200.0"),
    }

    chiX,tX,chiX_u,tX_u,chiX_uv,tX_uv = chi_and_derivs_updated(n, "X")
    chiY,tY,chiY_u,tY_u,chiY_uv,tY_uv = chi_and_derivs_updated(n, "Y")

    def fill(varname, chi, tchi, chi_u, t_u, chi_uv, t_uv):
        r  = idx[varname]
        val = g[varname] * chi * tchi
        dg[r] = val

        # log-diff for h = chi * tchi:
        # ln h = ln chi + ln tchi
        lnchi_u = chi_u / chi
        lnt_u   = t_u   / tchi
        Lambda_u = lnchi_u + lnt_u

        lnchi_uv = chi_uv/chi - np.outer(chi_u, chi_u)/(chi*chi)
        lnt_uv   = t_uv/tchi  - np.outer(t_u,   t_u)/(tchi*tchi)
        Lambda_uv = lnchi_uv + lnt_uv

        dg_p[r, :] = val * Lambda_u
        dg_pq[r, :, :] = val * (np.outer(Lambda_u, Lambda_u) + Lambda_uv)

    for nm in ["AA_XX","AB_XX","BB_XX"]:
        fill(nm, chiX,tX,chiX_u,tX_u,chiX_uv,tX_uv)
    for nm in ["AA_YY","AB_YY","BB_YY"]:
        fill(nm, chiY,tY,chiY_u,tY_u,chiY_uv,tY_uv)

    return dg, dg_p, dg_pq

def P_Q_and_derivs(n):
    """
    Toy Eq. 17 prefactors with all Z=1:

    For l=k (anion-diagonal) terms (XX and YY):
      P_{ij/XX} = 0.5 * n_{ij/XY}
      P_{ij/YY} = 0.5 * n_{ij/XY}

    For j=i (cation-diagonal) terms:
      Q_{AA/an} = 0.5 * n_{AB/an}
      Q_{BB/an} = 0.5 * n_{AB/an}
      for an in {XX,XY,YY}

    Returns P,Q and their first derivatives wrt all n_p.
    """
    P = np.zeros(M, dtype=DT); Q = np.zeros(M, dtype=DT)
    Pp = np.zeros((M, M), dtype=DT); Qp = np.zeros((M, M), dtype=DT)

    # P
    for ij in ["AA","AB","BB"]:
        ixy = idx[f"{ij}_XY"]
        for kk in ["XX","YY"]:
            r = idx[f"{ij}_{kk}"]
            P[r] = DT("0.5") * n[ixy]
            Pp[r, ixy] = DT("0.5")

    # Q
    for an in ["XX","XY","YY"]:
        iAB = idx[f"AB_{an}"]
        for ii in ["AA","BB"]:
            r = idx[f"{ii}_{an}"]
            Q[r] = DT("0.5") * n[iAB]
            Qp[r, iAB] = DT("0.5")

    return P, Q, Pp, Qp

def G_ex(n):
    """
    Toy MQMQA excess Gibbs energy (Eq. 17 structure):
      G^ex = 0.5*( T1 + T2 + T3 )

      T1 = sum_r n_r Δg_r
      T2 = sum_{l=k} P_r Δg_r
      T3 = sum_{j=i} Q_r Δg_r
    """
    n = asDT(n)
    if np.any(n <= 0):
        raise ValueError("All n must be > 0 for this toy.")

    dg,_,_ = delta_g_and_derivs(n)
    P,Q,_,_ = P_Q_and_derivs(n)

    T1 = np.dot(n, dg)

    diag_l_eq_k = [idx[v] for v in ["AA_XX","AB_XX","BB_XX","AA_YY","AB_YY","BB_YY"]]
    T2 = np.dot(P[diag_l_eq_k], dg[diag_l_eq_k])

    diag_j_eq_i = [idx[v] for v in ["AA_XX","BB_XX","AA_XY","BB_XY","AA_YY","BB_YY"]]
    T3 = np.dot(Q[diag_j_eq_i], dg[diag_j_eq_i])

    return DT("0.5") * (T1 + T2 + T3)

def H_ex_analytic(n):
    """
    Analytic Hessian of G^ex using the Eq.17 product-rule assembly:

      T1_pq = Δg_{p,q} + Δg_{q,p} + sum_r n_r Δg_{r,pq}

      T2_pq = sum_{l=k} [ P_{r,p}Δg_{r,q} + P_{r,q}Δg_{r,p} + P_rΔg_{r,pq} ]
              (P_{,pq}=0 since P linear in n)

      T3_pq = sum_{j=i} [ Q_{r,p}Δg_{r,q} + Q_{r,q}Δg_{r,p} + Q_rΔg_{r,pq} ]
              (Q_{,pq}=0)

      H^ex = 0.5*(T1_pq + T2_pq + T3_pq)
    """
    n = asDT(n)
    dg, dg_p, dg_pq = delta_g_and_derivs(n)
    P,Q,Pp,Qp = P_Q_and_derivs(n)

    H = np.zeros((M, M), dtype=DT)

    diag_l_eq_k = [idx[v] for v in ["AA_XX","AB_XX","BB_XX","AA_YY","AB_YY","BB_YY"]]
    diag_j_eq_i = [idx[v] for v in ["AA_XX","BB_XX","AA_XY","BB_XY","AA_YY","BB_YY"]]

    for p in range(M):
        for q in range(M):
            term = DT("0.0")

            # T1_pq
            term += dg_p[p, q] + dg_p[q, p]
            term += np.dot(n, dg_pq[:, p, q])

            # T2_pq
            for r in diag_l_eq_k:
                term += Pp[r, p]*dg_p[r, q] + Pp[r, q]*dg_p[r, p] + P[r]*dg_pq[r, p, q]

            # T3_pq
            for r in diag_j_eq_i:
                term += Qp[r, p]*dg_p[r, q] + Qp[r, q]*dg_p[r, p] + Q[r]*dg_pq[r, p, q]

            H[p, q] = DT("0.5") * term

    return H

def H_fd(n0, h):
    """
    Central FD Hessian on scalar G^ex.
    """
    n0 = asDT(n0)
    h = DT(h)
    if np.min(n0) <= h:
        raise ValueError("h too large: n0-h must stay positive for all components.")

    H = np.zeros((M, M), dtype=DT)
    G0 = G_ex(n0)

    # diagonal
    for p in range(M):
        e = np.zeros(M, dtype=DT); e[p] = DT("1.0")
        Gp = G_ex(n0 + h*e)
        Gm = G_ex(n0 - h*e)
        H[p, p] = (Gp - DT("2.0") * G0 + Gm) / (h*h)

    # off-diagonal
    for p in range(M):
        ep = np.zeros(M, dtype=DT); ep[p] = DT("1.0")
        for q in range(p+1, M):
            eq = np.zeros(M, dtype=DT); eq[q] = DT("1.0")
            Gpp = G_ex(n0 + h*ep + h*eq)
            Gpm = G_ex(n0 + h*ep - h*eq)
            Gmp = G_ex(n0 - h*ep + h*eq)
            Gmm = G_ex(n0 - h*ep - h*eq)
            val = (Gpp - Gpm - Gmp + Gmm) / (DT("4.0")*h*h)
            H[p, q] = val
            H[q, p] = val

    return H

def fro_norm(M):
    M = asDT(M)
    return np.sqrt(np.sum(M * M, dtype=DT))

# ==========================
# RUN THE VERIFICATION
# ==========================
if __name__ == "__main__":
    # Base state (all positive)
    n0 = np.array([DT("1.0"), DT("0.8"), DT("0.6"),
                   DT("0.7"), DT("0.4"), DT("0.3"),
                   DT("0.9"), DT("0.7"), DT("0.5")], dtype=DT)

    print("Variables (index order):")
    for i,v in enumerate(vars_):
        print(f"  {i:2d}: {v}")

    print("\nBase G^ex(n0) =", G_ex(n0))

    Ha = H_ex_analytic(n0)
    print("||H_analytic||_F =", float(fro_norm(Ha)))

    for h in [DT("1e-2"), DT("1e-3"), DT("1e-4"), DT("1e-5"), DT("1e-6")]:
        if np.min(n0) <= h:
            print(f"\nSkipping h={h:g} (would make some n negative).")
            continue
        Hfd = H_fd(n0, h)
        rel_err = fro_norm(Hfd - Ha) / max(DT("1.0"), fro_norm(Ha))
        print(f"\nh = {float(h):g}   relative error = {float(rel_err):.3e}")

        # no symmetry error needed bc H is symmetric by construction in both analytic and FD

Variables (index order):
   0: AA_XX
   1: AB_XX
   2: BB_XX
   3: AA_XY
   4: AB_XY
   5: BB_XY
   6: AA_YY
   7: AB_YY
   8: BB_YY

Base G^ex(n0) = 865.9658688202127886
||H_analytic||_F = 1030.4483145373927

h = 0.01   relative error = 3.079e-05

h = 0.001   relative error = 3.078e-07

h = 0.0001   relative error = 3.080e-09

h = 1e-05   relative error = 3.152e-09

h = 1e-06   relative error = 3.260e-07
